# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyansh-rath18/flyrank-internship-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Signal checks first (see code cell below for the bucket tables):**

- Signal 1 - staleness behind the refresh flag's assumption: stale (days_since_last_update >= 180) AND visible (impressions_90d >= 500) pages decline far more often (94.1%, n=17) than every other page (54.2%, n=29,983). Verdict: CONFIRMED - directionally strong, but n=17 is tiny, so treat the exact magnitude as directional, not precise.
- Signal 2 - CTR vs position behind the CTR-fix flag's assumption: mean CTR falls monotonically as position tier worsens - top_3 2.76% (n=1,116), page_1 0.65% (n=11,814), striking 0.32% (n=7,304), page_3_5 0.22% (n=7,242), deep 0.15% (n=1,319). Verdict: CONFIRMED, and well powered (n=28,795 total).

**The rule, in plain words:** a page is worth reviewing first if it has not been touched in a long time (stale: days_since_last_update >= 180) and it still earns real search visibility (visible: impressions_90d >= 500). Old content nobody has revisited that is still pulling meaningful impressions is the classic "traffic being left on the table" case - and Signal 1 shows that exact combination declines almost twice as often as everything else.

**Reason code it can output:** one reason code, `stale_visible_page`, for anything that matches both conditions; everything else carries `no_flag`.

In [5]:
import pandas as pd, numpy as np, os
REPO_DIR = '/content/flyrank-internship-'
import subprocess
if not os.path.exists(REPO_DIR): subprocess.run(['git', 'clone', '-q', 'https://github.com/Priyansh-rath18/flyrank-internship-.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print('rows:', len(df))
df['stale_visible'] = (df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)
df['declining'] = df['trend_direction'] == 'down'
bucket1 = df.groupby('stale_visible')['declining'].agg(['size', 'mean']).rename(columns={'size': 'n', 'mean': 'decline_rate'}).round(3)
print('\nSignal 1 -- staleness behind the refresh flag: stale_visible vs decline rate')
print(bucket1)
verdict1 = 'CONFIRMED' if bucket1.loc[True, 'decline_rate'] > bucket1.loc[False, 'decline_rate'] else 'OPPOSITE'
print('Verdict 1:', verdict1, '| n_stale_visible =', int(bucket1.loc[True, 'n']), '| n_other =', int(bucket1.loc[False, 'n']))
order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
bucket2 = df[df['avg_position'] > 0].groupby('position_tier')['ctr'].agg(['size', 'mean']).rename(columns={'size': 'n', 'mean': 'mean_ctr'}).reindex(order).round(3)
print('\nSignal 2 -- CTR vs position behind the CTR-fix flag: position_tier vs mean CTR')
print(bucket2)
ctr_vals = bucket2['mean_ctr'].dropna().tolist()
is_monotonic = all(ctr_vals[i] >= ctr_vals[i + 1] for i in range(len(ctr_vals) - 1))
verdict2 = 'CONFIRMED' if is_monotonic else ('MIXED' if ctr_vals[0] > ctr_vals[-1] else 'OPPOSITE')
print('Verdict 2:', verdict2, '| n total =', int(bucket2['n'].fillna(0).sum()))


cwd: /content/flyrank-internship-
rows: 30000

Signal 1 -- staleness behind the refresh flag: stale_visible vs decline rate
                   n  decline_rate
stale_visible                     
False          29983         0.542
True              17         0.941
Verdict 1: CONFIRMED | n_stale_visible = 17 | n_other = 29983

Signal 2 -- CTR vs position behind the CTR-fix flag: position_tier vs mean CTR
                   n  mean_ctr
position_tier                 
top_3           1116     2.764
page_1         11814     0.652
striking        7304     0.323
page_3_5        7242     0.222
deep            1319     0.150
Verdict 2: CONFIRMED | n total = 28795


## 2. Build the ranked queue (writes the CSV)

**Score:** `score = stale x visible x impressions_90d` (0 for anything that is not both stale and visible - readable on purpose, no fitted weights). **Reason code:** `stale_visible_page` when both conditions hold, else `no_flag`. **Action label:** `refresh` when flagged, else `monitor`.

Of 30,000 pages, 17 are flagged (matches Signal 1's n exactly, since it is the same rule). All 17 are from a single client - noted as a concentration caveat in section 4. The queue is written to `work/outputs/baseline_action_score.csv`, ranked by score descending.

In [6]:
stale = (df['days_since_last_update'] >= 180).astype(int)
visible = (df['impressions_90d'] >= 500).astype(int)
flagged = (stale & visible).astype(bool)
df['score'] = stale * visible * df['impressions_90d']
df['reason_code'] = np.where(flagged, 'stale_visible_page', 'no_flag')
df['action'] = np.where(flagged, 'refresh', 'monitor')
ranked = df.sort_values('score', ascending=False).reset_index(drop=True)
ranked['rank'] = ranked.index + 1
out_cols = ['rank', 'content_id', 'client_id', 'score', 'reason_code', 'action', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'trend_direction']
out = ranked[out_cols]
os.makedirs('work/outputs', exist_ok=True)
out.to_csv('work/outputs/baseline_action_score.csv', index=False)
print('wrote', len(out), 'rows to work/outputs/baseline_action_score.csv')
print('flagged rows:', int(flagged.sum()))
print(out.head(10).to_string(index=False))


wrote 30000 rows to work/outputs/baseline_action_score.csv
flagged rows: 17
 rank           content_id         client_id  score        reason_code  action  days_since_last_update  impressions_90d  avg_position  ctr trend_direction
    1 content_cf56e2e2e282 client_7f2253d7e2  61678 stale_visible_page refresh                     194            61678          19.7 0.15            down
    2 content_7368877ea310 client_7f2253d7e2  59472 stale_visible_page refresh                     194            59472          24.8 0.13            down
    3 content_1bfaa38ff26c client_7f2253d7e2  25715 stale_visible_page refresh                     194            25715          22.2 0.23            down
    4 content_0a91db491d14 client_7f2253d7e2  13299 stale_visible_page refresh                     193            13299          10.5 0.49            down
    5 content_5feee3994adb client_7f2253d7e2   7812 stale_visible_page refresh                     194             7812          39.0 0.01           

## 3. Top-10 review

All 10 top-ranked pages share the same reason code (`stale_visible_page`) and action (`refresh`), since only 17 pages in this 30k-row slice pass both thresholds and they are all sorted by raw impressions. Reading the printed table below, row by row: action = `refresh` for every row; why it's there = stale (>=180 days since update) and still visible (impressions_90d >= 500); confidence note = "high volume" for the top rows (impressions in the thousands to tens of thousands) tapering toward "moderate volume" further down; what would make it wrong = the `days_since_last_update` field being stale metadata rather than a real gap, or the traffic pattern being seasonal/consolidating rather than a genuine decay that a refresh would fix.

In [7]:
top10 = ranked.head(10).copy()
top10['why_its_there'] = np.where(top10['reason_code'] == 'stale_visible_page', 'stale (>=180d since update) and still visible (>=500 impressions/90d) - exactly the rule condition', 'did not meet the rule threshold')
top10['confidence_note'] = np.where(top10['impressions_90d'] >= 5000, 'high volume - directional confidence is decent', 'moderate volume - treat as directional, not precise')
top10['what_would_make_it_wrong'] = 'wrong if days_since_last_update is stale metadata (content actually changed recently), or if the traffic is seasonal/consolidating and will move without any refresh'
review_cols = ['rank', 'content_id', 'client_id', 'action', 'reason_code', 'confidence_note', 'impressions_90d', 'days_since_last_update', 'avg_position', 'what_would_make_it_wrong']
print(top10[review_cols].to_string(index=False))


 rank           content_id         client_id  action        reason_code                                     confidence_note  impressions_90d  days_since_last_update  avg_position                                                                                                                                             what_would_make_it_wrong
    1 content_cf56e2e2e282 client_7f2253d7e2 refresh stale_visible_page      high volume - directional confidence is decent            61678                     194          19.7 wrong if days_since_last_update is stale metadata (content actually changed recently), or if the traffic is seasonal/consolidating and will move without any refresh
    2 content_7368877ea310 client_7f2253d7e2 refresh stale_visible_page      high volume - directional confidence is decent            59472                     194          24.8 wrong if days_since_last_update is stale metadata (content actually changed recently), or if the traffic is seasonal/consolidating an

## 4. Weak picks + leakage check

**Weak pick:** none of the top 10 are already in `top_3` position, so no pick fails the "already winning, don't bother" check. The real weak spot is concentration - all 17 flagged pages, and therefore the whole top 10, belong to a single client (`client_7f2253d7e2`). A reviewer following this queue literally would spend 100% of their limited capacity on one client's pages, even though 31 other clients are in the data. That is a legitimate "looks wrong" result: the rule is honest about what it found, but a real rollout would need a per-client cap or a separate queue per client.

**Leakage check:** the score is built only from `days_since_last_update` and `impressions_90d` - both observed before any decision point. `trend_pct`, `trend_direction`, and `is_declining_label` (future-window/label-derived fields) are never inputs to the score, reason code, or action - confirmed programmatically below (the overlap between score inputs and label-derived columns is `set()` - empty). `trend_direction` is kept in the output CSV purely as read-only context for a human reviewer, the same way Signal 1 used it to check the rule, never as a feature.

In [8]:
weak = top10[(top10['avg_position'] > 0) & (top10['avg_position'] <= 3)]
print('Weak picks in the top 10 (already ranking top_3 - a refresh may add little):', len(weak))
print(weak[['rank', 'content_id', 'avg_position', 'impressions_90d']].to_string(index=False)) if len(weak) else print('none by that check')
client_counts = top10['client_id'].value_counts()
print('\nClient concentration in the top 10:')
print(client_counts)
print('-> weak pick pattern: the entire top 10 (and all 17 flagged rows) belong to one client, so this queue is not diversified across the portfolio - a reviewer following it strictly would spend all their time on one client.')
score_inputs = {'days_since_last_update', 'impressions_90d'}
future_or_label_cols = {'trend_pct', 'trend_direction', 'is_declining_label'}
print('\nLeakage check')
print('Columns that drive score/rank/reason_code/action:', score_inputs)
print('Future-window or label-derived columns (never used as score inputs):', future_or_label_cols)
print('Overlap between the two (should be empty):', score_inputs & future_or_label_cols)
print('trend_direction is only carried in the output CSV for human context, never used in the score calculation above.')


Weak picks in the top 10 (already ranking top_3 - a refresh may add little): 0
none by that check

Client concentration in the top 10:
client_id
client_7f2253d7e2    10
Name: count, dtype: int64
-> weak pick pattern: the entire top 10 (and all 17 flagged rows) belong to one client, so this queue is not diversified across the portfolio - a reviewer following it strictly would spend all their time on one client.

Leakage check
Columns that drive score/rank/reason_code/action: {'days_since_last_update', 'impressions_90d'}
Future-window or label-derived columns (never used as score inputs): {'is_declining_label', 'trend_direction', 'trend_pct'}
Overlap between the two (should be empty): set()
trend_direction is only carried in the output CSV for human context, never used in the score calculation above.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.